# 09 — Detailed Performance, Fairness, Calibration & Ablation

Extends the modeling with substantive secondary analyses (all patient-level, leakage-free, corrected
p-tau, 33 features): bootstrap-CI performance metrics, calibration (Brier), subgroup/fairness AUCs, and
feature-group ablation. Backs Results Tables 4.6–4.8 and Figure 5.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, brier_score_loss
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
RS=42; rng=np.random.RandomState(RS)
d=pd.read_csv('../data/pre_modelling_data.csv').sort_values(['PTID','Years.bl'])
g=d.groupby('PTID'); seqs=g['DX'].apply(list); nvis=g.size(); base=g.first().reset_index()
conf=lambda s,thr: any(s[i]>=thr and s[i+1]>=thr for i in range(len(s)-1))
demrev=lambda s: any(s[i]==2 and s[i+1]<2 for i in range(len(s)-1)); drop=set(seqs[seqs.apply(demrev)].index)
feat=[c for c in base.columns if c not in ['PTID','Years.bl','Month.bl','DX','DX_change_flag','Last_Visit_DX_Flag'] and not c.endswith('null_flag')]
print(len(feat),'features (PTAU included:', 'PTAU' in feat, ')')
def build(mask,thr):
    ids=[p for p in base[mask]['PTID'] if p not in drop and nvis[p]>=2]
    sub=base[base['PTID'].isin(ids)].reset_index(drop=True)
    return sub, pd.Series([int(conf(seqs[p],thr)) for p in sub['PTID']])
def oof(sub,y,cols=None):
    cols=cols or feat; X=sub[cols]; skf=StratifiedKFold(5,shuffle=True,random_state=RS); p=np.zeros(len(y))
    for tr,te in skf.split(X,y):
        pipe=ImbPipeline([('sc',StandardScaler()),('sm',SMOTE(random_state=RS)),('clf',RandomForestClassifier(n_estimators=400,max_depth=10,class_weight='balanced',random_state=RS,n_jobs=-1))])
        pipe.fit(X.iloc[tr],y.iloc[tr]); p[te]=pipe.predict_proba(X.iloc[te])[:,1]
    return p
cohorts={'CN→progression':build(base.DX==0,1),'MCI→Dementia':build(base.DX==1,2),'Pooled→AD':build(base.DX.isin([0,1]),2)}

33 features (PTAU included: True )


## Table 4.6 — Detailed metrics with bootstrap 95% CI + calibration

In [2]:
def boot(y,p,n=1000):
    a=[]
    for _ in range(n):
        i=rng.randint(0,len(y),len(y))
        if len(np.unique(y[i]))>1: a.append(roc_auc_score(y[i],p[i]))
    return np.percentile(a,[2.5,97.5])
store={}
print(f"{'Cohort':16s}{'AUC (95% CI)':20s}{'Sens':>6s}{'Spec':>6s}{'PPV':>6s}{'NPV':>6s}{'F1':>6s}{'Brier':>7s}")
for nm,(sub,ys) in cohorts.items():
    y=ys.values; p=oof(sub,ys); store[nm]=(sub,y,p)
    lo,hi=boot(y,p); pr=(p>=.5).astype(int); tn,fp,fn,tp=confusion_matrix(y,pr).ravel()
    se=tp/(tp+fn); sp=tn/(tn+fp); pv=tp/(tp+fp) if tp+fp else 0; nv=tn/(tn+fn) if tn+fn else 0
    print(f"{nm:16s}{roc_auc_score(y,p):.2f} ({lo:.2f}-{hi:.2f})   {se*100:5.0f}{sp*100:6.0f}{pv*100:6.0f}{nv*100:6.0f}{2*pv*se/(pv+se):6.2f}{brier_score_loss(y,p):7.3f}")

Cohort          AUC (95% CI)          Sens  Spec   PPV   NPV    F1  Brier


CN→progression  0.65 (0.58-0.72)      24    89    28    88  0.26  0.147


MCI→Dementia    0.82 (0.80-0.85)      65    80    56    86  0.60  0.155


Pooled→AD       0.88 (0.86-0.90)      67    88    55    92  0.61  0.113


## Table 4.7 — Subgroup / fairness AUC (Pooled → AD)

In [3]:
sub,y,p=store['Pooled→AD']; s=sub.copy(); s['y']=y; s['p']=p
def a(mask,lab):
    m=mask.values; yy=s['y'][m].values; pp=s['p'][m].values
    print(f"  {lab:14s} n={m.sum():4d}  events={int(yy.sum()):3d}  AUC={roc_auc_score(yy,pp):.2f}")
for mask,lab in [(s.PTGENDER==1,'Male'),(s.PTGENDER==0,'Female'),(s.APOE4==0,'APOE4-neg'),(s.APOE4>0,'APOE4-pos'),
                 (s.PTEDUCAT<16,'Educ <16y'),(s.PTEDUCAT>=16,'Educ >=16y'),(s.AGE<75,'Age <75'),(s.AGE>=75,'Age >=75')]:
    a(mask,lab)

  Male           n= 737  events=142  AUC=0.87
  Female         n= 601  events=102  AUC=0.88
  APOE4-neg      n= 771  events= 83  AUC=0.87
  APOE4-pos      n= 567  events=161  AUC=0.84
  Educ <16y      n= 447  events= 92  AUC=0.83
  Educ >=16y     n= 891  events=152  AUC=0.90
  Age <75        n= 767  events=134  AUC=0.91
  Age >=75       n= 571  events=110  AUC=0.82


## Table 4.8 — Feature-group ablation (Pooled → AD)

In [4]:
sub,ys=cohorts['Pooled→AD']; y=ys
groups={'Cognitive':['ADAS13','MMSE','MOCA','CDRSB','FAQ','mPACCdigit','mPACCtrailsB','LDELTOTAL','RAVLT.immediate','RAVLT.learning','RAVLT.forgetting','RAVLT.perc.forgetting','TRABSCOR'],
 'MRI':['Entorhinal','Fusiform','Hippocampus','ICV','MidTemp','Ventricles','WholeBrain'],'PET':['AV45','FDG'],
 'CSF':['ABETA','TAU','PTAU'],'Demographic':['AGE','PTEDUCAT','PTGENDER','APOE4','Married','Widowed','Divorced','Never_married']}
def cvauc(cols):
    X=sub[cols]; skf=StratifiedKFold(5,shuffle=True,random_state=RS); aa=[]
    for tr,te in skf.split(X,y):
        pipe=ImbPipeline([('sc',StandardScaler()),('sm',SMOTE(random_state=RS)),('clf',RandomForestClassifier(n_estimators=400,max_depth=10,class_weight='balanced',random_state=RS,n_jobs=-1))])
        pipe.fit(X.iloc[tr],y.iloc[tr]); aa.append(roc_auc_score(y.iloc[te],pipe.predict_proba(X.iloc[te])[:,1]))
    return np.mean(aa)
full=cvauc(feat); print(f"Full-model AUC={full:.3f}\n{'Group':13s}{'alone':>8s}{'removed':>9s}{'Δremove':>9s}")
for gn,gf in groups.items():
    gf=[f for f in gf if f in feat]
    print(f"{gn:13s}{cvauc(gf):>8.3f}{cvauc([f for f in feat if f not in gf]):>9.3f}{full-cvauc([f for f in feat if f not in gf]):>+9.3f}")

Full-model AUC=0.880
Group           alone  removed  Δremove


Cognitive       0.855    0.838   +0.042


MRI             0.747    0.877   +0.003


PET             0.791    0.879   +0.001


CSF             0.755    0.877   +0.003


Demographic     0.659    0.876   +0.004


## Summary
- **Table 4.6:** MCI→Dementia AUC 0.82 (0.80–0.85), Pooled 0.88 (0.86–0.90); high NPV (86–92%), Brier 0.11–0.16.
- **Table 4.7:** equitable by sex; lower AUC for less-educated (0.83), older (0.82 at ≥75), and APOE4+ (0.84).
- **Table 4.8:** cognitive/functional measures dominate (alone 0.855; −0.042 if removed); other modalities redundant.